In [39]:
!git clone https://github.com/Tahoni01/Continual-hate-speech-detection.git
%cd Continual-hate-speech-detection
!ls

Cloning into 'Continual-hate-speech-detection'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 86 (delta 30), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 174.86 KiB | 4.60 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/Continual-hate-speech-detection
dataset  Main.ipynb  models  old_main.ipynb  README.md	requirements.txt  src


In [40]:
import torch

from dataset.df_loader import (
    getdf_davidson,
    getdf_hatexplain
)

from dataset.stream_generator import (
    create_stream,
    merge_streams,
    online_stream,
    stream_summary,
    dataset_distribution
)

from transformers import (
    AutoTokenizer,
    AutoConfig
)

from models.model_builder import CustomClassifier

In [41]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

DEVICE: cpu


In [42]:
print("Loading datasets...")

df_dv = getdf_davidson()
df_hx = getdf_hatexplain()

print("Davidson:", df_dv.shape)
print("HateXplain:", df_hx.shape)

Loading datasets...
                                                text      label    source
0  rt as a woman you shouldnt complain about clea...     normal  davidson
1  rt boy dats coldtyga dwn bad for cuffin dat ho...  offensive  davidson
2  rt dawg rt you ever fuck a bitch and she start...  offensive  davidson
3                          rt she look like a tranny  offensive  davidson
4  rt the shit you hear about me might be true or...  offensive  davidson
                                                text       label      source
0  i dont think im getting my baby them white 9 h...      normal  hatexplain
1  we cannot continue calling ourselves feminists...      normal  hatexplain
2                      nawt yall niggers ignoring me      normal  hatexplain
3  user i am bit confused coz chinese ppl can not...  hatespeech  hatexplain
4  this bitch in whataburger eating a burger with...  hatespeech  hatexplain
Davidson: (24275, 3)
HateXplain: (20144, 3)


In [43]:
dv_stream = create_stream(
    df=df_dv,
    batch_size=32,
    shuffle=True
)

hx_stream = create_stream(
    df=df_hx,
    batch_size=32,
    shuffle=True
)

In [44]:
stream_summary(dv_stream, "Davidson Stream")
stream_summary(hx_stream, "HateXplain Stream")


STREAM SUMMARY: Davidson Stream
------------------------------
total batches: 759
total samples: 24275
avg batch size: 31.98

STREAM SUMMARY: HateXplain Stream
------------------------------
total batches: 630
total samples: 20144
avg batch size: 31.97


In [45]:
full_stream = merge_streams(
    streams=[dv_stream, hx_stream],
    shuffle_streams=False
)

stream_summary(full_stream, "FULL STREAM")
dataset_distribution(full_stream)


STREAM SUMMARY: FULL STREAM
------------------------------
total batches: 1389
total samples: 44419
avg batch size: 31.98

LABEL DISTRIBUTION
------------------------------
label
offensive     0.547356
normal        0.267386
hatespeech    0.185259
Name: proportion, dtype: float64


In [46]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [47]:
num_labels = 2

config = AutoConfig.from_pretrained(
    "roberta-base",
    num_labels=num_labels
)

In [48]:
model = CustomClassifier(
    model_name="roberta-base",
    config=config,
    class_weights=None,
    use_lora=True
).to(device)

print("Model loaded")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded


In [49]:
print("Starting stream simulation...\n")

stream_iterator = online_stream(full_stream)

for step, batch in enumerate(stream_iterator):

    print("=" * 50)
    print(f"STREAM STEP {step}")
    print("=" * 50)

    print(batch.head())

    print("\nBatch size:", len(batch))

    print("\nLabel distribution:")
    print(batch["label"].value_counts())

    # ---------------------------------
    # FUTURE:
    # train_step(batch)
    # ---------------------------------

    if step == 2:
        break

Starting stream simulation...

STREAM STEP 0
                                                text       label    source
0  this that bitch nigga shit they invented it i ...   offensive  davidson
1  she made fun of a suicide attempt mellie aint ...   offensive  davidson
2  and i cant be known for fucking wit a trash bitch   offensive  davidson
3  im wit yo bitch smokin let her keep the mid im...   offensive  davidson
4                              chill ur pussy gay af  hatespeech  davidson

Batch size: 32

Label distribution:
label
offensive     23
normal         6
hatespeech     3
Name: count, dtype: int64
STREAM STEP 1
                                                 text      label    source
32  tattoos are about expression not meaning one m...     normal  davidson
33                      i want me a coon ass girl lol  offensive  davidson
34  marilyn monroe set the standard for what hoes ...  offensive  davidson
35  think i should change 9733doctors advocate9733...  offensive  david